In [ ]:
COLLECTION_MAPPING = [
    # ('gen_ml_s100_d20_sp501', 'test_ml_s600_d20_sp0.csv'),  # test
    # ('gen_ml_s100_d20_sp50', 'test_ml_s600_d20_sp0.csv'),  # test
    # ('gen_ml_s100_d20_sp50', 'test_ml_s600_d20_sp0.csv'),  # test


    ('news_real_embedings_s69878', 'test_news_s600_d384_sp0.csv'),  # REAL NEWS
    
    ('gen_news_s69878_d384_sp0', 'test_news_s600_d384_sp0.csv'),  # test dim for news
    ('gen_news_s69878_d3000_sp0', 'test_news_s600_d3000_sp0.csv'),
    ('gen_news_s69878_d5000_sp0', 'test_news_s600_d5000_sp0.csv'),
    # ('gen_news_s69878_d7000_sp0', 'test_news_s600_d7000_sp0.csv'),
    # ('gen_news_s69878_d9000_sp0', 'test_news_s600_d9000_sp0.csv'),
    # ('gen_news_s69878_d10677_sp0', 'test_news_s600_d10677_sp0.csv'),

    # ('gen_ml_s69878_d384_sp0', 'test_ml_s600_d384_sp0.csv'),  # test dim for ml
    # ('gen_ml_s69878_d3000_sp0', 'test_ml_s600_d3000_sp0.csv'),
    # ('gen_ml_s69878_d5000_sp0', 'test_ml_s600_d5000_sp0.csv'),
    # ('gen_ml_s69878_d7000_sp0', 'test_ml_s600_d7000_sp0.csv'),
    # ('gen_ml_s69878_d9000_sp0', 'test_ml_s600_d9000_sp0.csv'),
    # ('gen_ml_s69878_d10677_sp0', 'test_ml_s600_d10677_sp0.csv'),

    # ('ml_real_pivoted_na_s69878', 'test_ml_s600_d10677_sp0.csv'),  # REAL ML
    
    # ('gen_ml_s69878_d10677_sp10', 'test_ml_s600_d10677_sp0.csv'),  # test sparsity for ml
    # ('gen_ml_s69878_d10677_sp20', 'test_ml_s600_d10677_sp0.csv'),
    # ('gen_ml_s69878_d10677_sp30', 'test_ml_s600_d10677_sp0.csv'),
    # ('gen_ml_s69878_d10677_sp40', 'test_ml_s600_d10677_sp0.csv'),
    # ('gen_ml_s69878_d10677_sp50', 'test_ml_s600_d10677_sp0.csv'),
    # ('gen_ml_s69878_d10677_sp60', 'test_ml_s600_d10677_sp0.csv'),
    # ('gen_ml_s69878_d10677_sp70', 'test_ml_s600_d10677_sp0.csv'),  # fail
    # ('gen_ml_s69878_d10677_sp80', 'test_ml_s600_d10677_sp0.csv'),
    # ('gen_ml_s69878_d10677_sp90', 'test_ml_s600_d10677_sp0.csv'),
    # ('gen_ml_s69878_d10677_sp98', 'test_ml_s600_d10677_sp0.csv'),

]

# Chroma 100nn

In [ ]:
# TEST CHROMA
import pandas as pd
import chromadb

from time import time

N_RESULTS = 100

def estimate_chroma(embeddings, db_name, n_results, client=None):
    chroma_client = client if client is not None else chromadb.PersistentClient(path='./data/chroma')
    collection = chroma_client.get_collection(db_name)
    # collection.count()
    # print(f'{db_name} {collection.count()}')
    st = time()
    for emb in embeddings:
        # print('.', end='')
        res = collection.query(query_embeddings=emb, n_results=n_results)
        # print(res['distances'][:10])
    # print()
    return res, str(time() - st).replace('.', ',')


# TEST CHROMA
chroma_client = chromadb.PersistentClient(path='./data/chroma')


for col_name, inp_file in COLLECTION_MAPPING:
    print(f'CHROMA_NN{N_RESULTS}')
    print(col_name)
    print(f'{inp_file=}')
    # Perform 6 estimations - 100 querises per attempt
    for i, batch in enumerate(pd.read_csv(f'./data/testing/{inp_file}', chunksize=100, header=None)):
        res, t = estimate_chroma(batch.values.tolist(), col_name, N_RESULTS, client=chroma_client)
        print(t)
    print()

In [ ]:
res

# Chroma 2500nn

In [ ]:
# TEST CHROMA
import pandas as pd
import chromadb

from time import time

N_RESULTS = 2500

def estimate_chroma(embeddings, db_name, n_results, client=None):
    chroma_client = client if client is not None else chromadb.PersistentClient(path='./data/chroma')
    collection = chroma_client.get_collection(db_name)
    # collection.count()
    # print(f'{db_name} {collection.count()}')
    st = time()
    for emb in embeddings:
        # print('.', end='')
        res = collection.query(query_embeddings=emb, n_results=n_results)
        # print(res['distances'][:10])
    # print()
    return res, str(time() - st).replace('.', ',')


# TEST CHROMA
chroma_client = chromadb.PersistentClient(path='./data/chroma')


for col_name, inp_file in COLLECTION_MAPPING[::-1]:
    print(f'CHROMA_NN{N_RESULTS}')
    print(col_name)
    print(f'{inp_file=}')
    # Perform 6 estimations - 100 querises per attempt
    for i, batch in enumerate(pd.read_csv(f'./data/testing_2500nn/{inp_file}', chunksize=100, header=None)):
        res, t = estimate_chroma(batch.values.tolist(), col_name, N_RESULTS, client=chroma_client)
        print(t)
    print()

In [ ]:
res

# Qdrant 100nn

In [ ]:
# TEST QDRANT
import pandas as pd
from qdrant_client import QdrantClient

from time import time


def estimate_qdrant(embeddings, db_name, n_results, client):
    st = time()
    for emb in embeddings:
        # print('.', end='')
        res = client.query_points(
            collection_name=db_name,
            query=emb,
            limit=n_results,
            with_vectors=False,  # Change to False, since we don't return vectors in Chroma
        )
    # print()
    return res, str(time() - st).replace('.', ',')

# TEST QDRANT
for col_name, inp_file in COLLECTION_MAPPING:
    print(f'QDRANT_NN{N_RESULTS}')
    print(col_name)
    print(f'{inp_file=}')
    print('init client...')
    st_cl = time()
    client = QdrantClient(path=f"./data/qdrant/sep_clients/{col_name}")
    # client = QdrantClient(path=f"./data/qdrant")
    print(str(time() - st_cl).replace('.', ','))
    print('done.')
    # Perform 6 estimations - 100 querises per attempt
    for i, batch in enumerate(pd.read_csv(f'./data/testing/{inp_file}', chunksize=100, header=None)):
        res, t = estimate_qdrant(batch.values.tolist(), col_name, N_RESULTS, client)
        print(t)

    print('closing client...')
    st = time()
    client.close()
    print(f'done in {time()-st} sec.')
    print()

In [ ]:
res.points

# Qdrant Docker 100nn

In [ ]:
# TEST QDRANT in Docker
import pandas as pd
from qdrant_client import QdrantClient

from time import time

N_RESULTS = 100

def estimate_qdrant(embeddings, db_name, n_results, client):
    st = time()
    for emb in embeddings:
        # print('.', end='')
        res = client.query_points(
            collection_name=db_name,
            query=emb,
            limit=n_results,
            with_vectors=False,  # Change to False, since we don't return vectors in Chroma
        )
    # print()
    return res, str(time() - st).replace('.', ',')


qd_client = QdrantClient(host="localhost", port=6333)

# TEST QDRANT
for col_name, inp_file in COLLECTION_MAPPING:
    print(f'QDRANT_NN{N_RESULTS}')
    print(col_name)
    print(f'{inp_file=}')
    
    # Perform 6 estimations - 100 querises per attempt
    for i, batch in enumerate(pd.read_csv(f'./data/testing/{inp_file}', chunksize=100, header=None)):
        # if i>= 3: continue  # Qdrand is runing very long, so perform only 3 estimations
        res, t = estimate_qdrant(batch.values.tolist(), col_name, N_RESULTS, qd_client)
        print(t)
    print()

In [ ]:
res.points

# Qdrant Docker 2500nn

In [ ]:
N_RESULTS = 2500
COLLECTION_MAPPING = [
    # ('gen_ml_s100_d20_sp501', 'test_ml_s600_d20_sp0.csv'),  # test
    # ('gen_ml_s100_d20_sp50', 'test_ml_s600_d20_sp0.csv'),  # test
    # ('gen_ml_s100_d20_sp50', 'test_ml_s600_d20_sp0.csv'),  # test


    # ('news_real_embedings_s69878', 'test_news_s600_d384_sp0.csv'),  # REAL NEWS  # done
    
    # ('gen_news_s69878_d384_sp0', 'test_news_s600_d384_sp0.csv'),  # test dim for news  # done
    # ('gen_news_s69878_d3000_sp0', 'test_news_s600_d3000_sp0.csv'),  # done
    # ('gen_news_s69878_d5000_sp0', 'test_news_s600_d5000_sp0.csv'),  # done
    # ('gen_news_s69878_d7000_sp0', 'test_news_s600_d7000_sp0.csv'),  # done
    # ('gen_news_s69878_d9000_sp0', 'test_news_s600_d9000_sp0.csv'),  # done
    # ('gen_news_s69878_d10677_sp0', 'test_news_s600_d10677_sp0.csv'),  # done

    # ('gen_ml_s69878_d384_sp0', 'test_ml_s600_d384_sp0.csv'),  # test dim for ml  # done
    # ('gen_ml_s69878_d3000_sp0', 'test_ml_s600_d3000_sp0.csv'),  # done
    # ('gen_ml_s69878_d5000_sp0', 'test_ml_s600_d5000_sp0.csv'),  # done
    # ('gen_ml_s69878_d7000_sp0', 'test_ml_s600_d7000_sp0.csv'),  # done
    # ('gen_ml_s69878_d9000_sp0', 'test_ml_s600_d9000_sp0.csv'),  # done
    # ('gen_ml_s69878_d10677_sp0', 'test_ml_s600_d10677_sp0.csv'),  # done

    # ('ml_real_pivoted_na_s69878', 'test_ml_s600_d10677_sp0.csv'),  # REAL ML  # done
    
    # ('gen_ml_s69878_d10677_sp10', 'test_ml_s600_d10677_sp0.csv'),  # test sparsity for ml  # done
    # ('gen_ml_s69878_d10677_sp20', 'test_ml_s600_d10677_sp0.csv'),  # done
    # ('gen_ml_s69878_d10677_sp30', 'test_ml_s600_d10677_sp0.csv'),  # done
    # ('gen_ml_s69878_d10677_sp40', 'test_ml_s600_d10677_sp0.csv'),  # done
    # ('gen_ml_s69878_d10677_sp50', 'test_ml_s600_d10677_sp0.csv'),  # done
    # ('gen_ml_s69878_d10677_sp60', 'test_ml_s600_d10677_sp0.csv'),  # done
    # ('gen_ml_s69878_d10677_sp70', 'test_ml_s600_d10677_sp0.csv'),  # done
    # ('gen_ml_s69878_d10677_sp80', 'test_ml_s600_d10677_sp0.csv'),  # done
    # ('gen_ml_s69878_d10677_sp90', 'test_ml_s600_d10677_sp0.csv'),  # done
    # ('gen_ml_s69878_d10677_sp98', 'test_ml_s600_d10677_sp0.csv'),  # done

]

In [ ]:
# TEST QDRANT in Docker
import pandas as pd
from qdrant_client import QdrantClient

from time import time


def estimate_qdrant(embeddings, db_name, n_results, client):
    st = time()
    for emb in embeddings:
        # print('.', end='')
        res = client.query_points(
            collection_name=db_name,
            query=emb,
            limit=n_results,
            with_vectors=False,  # Change to False, since we don't return vectors in Chroma
        )
    # print()
    return res, str(time() - st).replace('.', ',')


# qd_client = QdrantClient(host="localhost", port=6333, timeout=999999)

# TEST QDRANT
for col_name, inp_file in COLLECTION_MAPPING[::-1]:
    print(f'QDRANT_NN{N_RESULTS}')
    print(col_name)
    # print(f'{inp_file=}')

    hits = qd_client.query_points(
        collection_name=col_name,
        query=57901,
        # query=query_vector,
        limit=2,  # Return 5 closest points
        with_vectors=True,
        timeout=999999,
    )
    print('dimension=', len(hits.points[0].vector))
    # print(len(hits.points), *hits.points, sep='\n')
    
    # Perform 6 estimations - 100 querises per attempt
    for i, batch in enumerate(pd.read_csv(f'./data/testing_2500nn/{inp_file}', chunksize=100, header=None)):
        if i>= 3: continue  # Qdrand is runing very long, so perform only 3 estimations
        res, t = estimate_qdrant(batch.values.tolist(), col_name, N_RESULTS, qd_client)
        print(t)
    print()

In [5]:
qd_client.close()

# Qdrant Docker Retesting

In [ ]:
import time
sec = 120
print(f'sleep {sec} seconds...')
time.sleep(sec)
print('done')

In [ ]:
N_RESULTS = 100
COLLECTION_MAPPING = [
    # ('gen_ml_s100_d20_sp501', 'test_ml_s600_d20_sp0.csv'),  # test
    # ('gen_ml_s100_d20_sp50', 'test_ml_s600_d20_sp0.csv'),  # test
    # ('gen_ml_s100_d20_sp50', 'test_ml_s600_d20_sp0.csv'),  # test


    # ('news_real_embedings_s69878', 'test_news_s600_d384_sp0.csv'),  # REAL NEWS  # done twice
    
    # ('gen_news_s69878_d384_sp0', 'test_news_s600_d384_sp0.csv'),  # test dim for news  # done twice
    # ('gen_news_s69878_d3000_sp0', 'test_news_s600_d3000_sp0.csv'),  # done twice
    # ('gen_news_s69878_d5000_sp0', 'test_news_s600_d5000_sp0.csv'),  # done twice
    # ('gen_news_s69878_d7000_sp0', 'test_news_s600_d7000_sp0.csv'),  # done twice
    # ('gen_news_s69878_d9000_sp0', 'test_news_s600_d9000_sp0.csv'),  # done twice
    # ('gen_news_s69878_d10677_sp0', 'test_news_s600_d10677_sp0.csv'),  # done twice

    # ('gen_ml_s69878_d384_sp0', 'test_ml_s600_d384_sp0.csv'),  # test dim for ml  # done twice
    # ('gen_ml_s69878_d3000_sp0', 'test_ml_s600_d3000_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d5000_sp0', 'test_ml_s600_d5000_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d7000_sp0', 'test_ml_s600_d7000_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d9000_sp0', 'test_ml_s600_d9000_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d10677_sp0', 'test_ml_s600_d10677_sp0.csv'),  # done twice

    # ('ml_real_pivoted_na_s69878', 'test_ml_s600_d10677_sp0.csv'),  # REAL ML  # done twice
    
    # ('gen_ml_s69878_d10677_sp10', 'test_ml_s600_d10677_sp0.csv'),  # test sparsity for ml  # done twice
    # ('gen_ml_s69878_d10677_sp20', 'test_ml_s600_d10677_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d10677_sp30', 'test_ml_s600_d10677_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d10677_sp40', 'test_ml_s600_d10677_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d10677_sp50', 'test_ml_s600_d10677_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d10677_sp60', 'test_ml_s600_d10677_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d10677_sp70', 'test_ml_s600_d10677_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d10677_sp80', 'test_ml_s600_d10677_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d10677_sp90', 'test_ml_s600_d10677_sp0.csv'),  # done twice
    # ('gen_ml_s69878_d10677_sp98', 'test_ml_s600_d10677_sp0.csv'),  # done twice

]

import pandas as pd
from qdrant_client import QdrantClient
from time import time

def estimate_qdrant(embeddings, db_name, n_results, client):
    st = time()
    for emb in embeddings:
        # print('.', end='')
        res = client.query_points(
            collection_name=db_name,
            query=emb,
            limit=n_results,
            with_vectors=False,  # Change to False, since we don't return vectors in Chroma
            timeout=999999,
        )
    # print()
    return res, str(time() - st).replace('.', ',')

# TEST QDRANT in Docker
qd_client = QdrantClient(host="localhost", port=6333, timeout=999999)

# TEST QDRANT in Docker
for col_name, inp_file in COLLECTION_MAPPING[::-1]:
    print(f'QDRANT_NN{N_RESULTS}')
    print(col_name)
    # print(f'{inp_file=}')

    hits = qd_client.query_points(
        collection_name=col_name,
        query=57901,
        # query=query_vector,
        limit=2,  # Return 5 closest points
        with_vectors=True,
        timeout=999999,
    )
    print('dimension=', len(hits.points[0].vector))
    # print(len(hits.points), *hits.points, sep='\n')
    
    # Perform 6 estimations - 100 querises per attempt
    for i, batch in enumerate(pd.read_csv(f'./data/testing_2500nn/{inp_file}', chunksize=100, header=None)):
        # if i>= 3: continue  # Qdrand is runing very long, so perform only 3 estimations
        res, t = estimate_qdrant(batch.values.tolist(), col_name, N_RESULTS, qd_client)
        print(t)
    print()

In [5]:
qd_client.close()